In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a single GPT-2 transformer decoder block. Given an input tensor
  $x$ of shape <code>(seq_len, 768)</code> and a packed weight buffer containing
  all block parameters, compute the output using pre-norm architecture with
  multi-head self-attention and a feed-forward network with GELU activation.
</p>

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 320 600" width="320" height="600" style="display:block; margin:20px auto;">
  <defs>
    <marker id="ah" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M0 0L10 5L0 10z" fill="#999"/>
    </marker>
  </defs>

  <!-- Input label -->
  <text x="130" y="18" text-anchor="middle" fill="#ccc" font-size="13" font-family="monospace">x (seq_len, 768)</text>

  <!-- Arrow: input -> LN1 -->
  <line x1="130" y1="26" x2="130" y2="44" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Residual 1: fork right, down, back left to Add1 -->
  <line x1="130" y1="33" x2="260" y2="33" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4"/>
  <line x1="260" y1="33" x2="260" y2="270" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4"/>
  <line x1="260" y1="270" x2="145" y2="270" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4" marker-end="url(#ah)"/>
  <text x="268" y="155" fill="#666" font-size="10" font-family="sans-serif" transform="rotate(90,268,155)">residual</text>

  <!-- LN1 -->
  <rect x="55" y="47" width="150" height="30" rx="5" fill="#333" stroke="#777" stroke-width="1"/>
  <text x="130" y="67" text-anchor="middle" fill="#ccc" font-size="12" font-family="sans-serif">LayerNorm 1</text>
  <line x1="130" y1="77" x2="130" y2="95" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- QKV Proj -->
  <rect x="55" y="98" width="150" height="30" rx="5" fill="#1e2d4d" stroke="#4477bb" stroke-width="1"/>
  <text x="130" y="118" text-anchor="middle" fill="#aaccee" font-size="12" font-family="sans-serif">QKV Projection</text>
  <line x1="130" y1="128" x2="130" y2="146" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- MHA -->
  <rect x="55" y="149" width="150" height="30" rx="5" fill="#1e2d4d" stroke="#4477bb" stroke-width="1"/>
  <text x="130" y="169" text-anchor="middle" fill="#aaccee" font-size="12" font-family="sans-serif">Multi-Head Attention</text>
  <line x1="130" y1="179" x2="130" y2="197" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Attn Out Proj -->
  <rect x="55" y="200" width="150" height="30" rx="5" fill="#1e2d4d" stroke="#4477bb" stroke-width="1"/>
  <text x="130" y="220" text-anchor="middle" fill="#aaccee" font-size="12" font-family="sans-serif">Output Projection</text>
  <line x1="130" y1="230" x2="130" y2="258" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Add 1 -->
  <circle cx="130" cy="270" r="12" fill="#222" stroke="#999" stroke-width="1.5"/>
  <text x="130" y="275" text-anchor="middle" fill="#ccc" font-size="15" font-family="sans-serif" font-weight="bold">+</text>
  <line x1="130" y1="282" x2="130" y2="306" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Residual 2: fork right, down, back left to Add2 -->
  <line x1="130" y1="292" x2="260" y2="292" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4"/>
  <line x1="260" y1="292" x2="260" y2="530" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4"/>
  <line x1="260" y1="530" x2="145" y2="530" stroke="#555" stroke-width="1.5" stroke-dasharray="5,4" marker-end="url(#ah)"/>
  <text x="268" y="415" fill="#666" font-size="10" font-family="sans-serif" transform="rotate(90,268,415)">residual</text>

  <!-- LN2 -->
  <rect x="55" y="309" width="150" height="30" rx="5" fill="#333" stroke="#777" stroke-width="1"/>
  <text x="130" y="329" text-anchor="middle" fill="#ccc" font-size="12" font-family="sans-serif">LayerNorm 2</text>
  <line x1="130" y1="339" x2="130" y2="357" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- FC -->
  <rect x="55" y="360" width="150" height="30" rx="5" fill="#1e3d2d" stroke="#44aa66" stroke-width="1"/>
  <text x="130" y="380" text-anchor="middle" fill="#aaeebb" font-size="12" font-family="sans-serif">Linear (768 → 3072)</text>
  <line x1="130" y1="390" x2="130" y2="408" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- GELU -->
  <rect x="55" y="411" width="150" height="30" rx="5" fill="#1e3d2d" stroke="#44aa66" stroke-width="1"/>
  <text x="130" y="431" text-anchor="middle" fill="#aaeebb" font-size="12" font-family="sans-serif">GELU</text>
  <line x1="130" y1="441" x2="130" y2="459" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Proj -->
  <rect x="55" y="462" width="150" height="30" rx="5" fill="#1e3d2d" stroke="#44aa66" stroke-width="1"/>
  <text x="130" y="482" text-anchor="middle" fill="#aaeebb" font-size="12" font-family="sans-serif">Linear (3072 → 768)</text>
  <line x1="130" y1="492" x2="130" y2="518" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Add 2 -->
  <circle cx="130" cy="530" r="12" fill="#222" stroke="#999" stroke-width="1.5"/>
  <text x="130" y="535" text-anchor="middle" fill="#ccc" font-size="15" font-family="sans-serif" font-weight="bold">+</text>
  <line x1="130" y1="542" x2="130" y2="566" stroke="#999" stroke-width="1.5" marker-end="url(#ah)"/>

  <!-- Output label -->
  <text x="130" y="584" text-anchor="middle" fill="#ccc" font-size="13" font-family="monospace">output (seq_len, 768)</text>
</svg>

<p>The block uses GPT-2's <strong>pre-norm</strong> architecture: LayerNorm is applied
<em>before</em> each sub-layer (attention and feed-forward), not after. At a high level:</p>

$$
\begin{aligned}
x' &= x + \text{MultiHeadAttn}\!\left(\text{LN}_1(x)\right) \$$4pt]
\text{output} &= x' + \text{FeedForward}\!\left(\text{LN}_2(x')\right)
\end{aligned}
$$

<p>where the sub-layers are defined as:</p>

$$
\begin{aligned}
\text{LN}(z) &= \frac{z - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta, \quad \mu = \frac{1}{d}\sum_i z_i, \quad \sigma^2 = \frac{1}{d}\sum_i (z_i - \mu)^2 \$$8pt]
[Q \mid K \mid V] &= \text{LN}_1(x) \cdot W_{qkv} + b_{qkv} \$$4pt]
\text{head}_i &= \text{softmax}\!\left(\frac{Q_i K_i^\top}{\sqrt{d_k}}\right) V_i, \quad d_k = 64 \$$4pt]
\text{MultiHeadAttn}(z) &= \text{Concat}(\text{head}_1, \ldots, \text{head}_{12}) \cdot W_{\text{attn}} + b_{\text{attn}} \$$8pt]
\text{FeedForward}(z) &= \text{GELU}\!\left(z \cdot W_{fc} + b_{fc}\right) \cdot W_{\text{proj}} + b_{\text{proj}}
\end{aligned}
$$

<p>Expanding into individual steps:</p>

<ol>
  <li><strong>Layer Norm 1:</strong> $x_{\text{norm}} = \text{LN}_1(x)$ with parameters $\gamma_1, \beta_1$</li>
  <li><strong>QKV Projection:</strong> $QKV = x_{\text{norm}} \cdot W_{qkv} + b_{qkv}$, split into $Q, K, V$ each of shape <code>(seq_len, 768)</code></li>
  <li><strong>Multi-Head Attention:</strong> Reshape $Q, K, V$ into 12 heads of dimension 64, compute per-head scaled dot-product attention (no causal mask), then concatenate heads into $A$</li>
  <li><strong>Output Projection:</strong> $P = A \cdot W_{\text{attn}} + b_{\text{attn}}$</li>
  <li><strong>Residual 1:</strong> $x' = x + P$</li>
  <li><strong>Layer Norm 2:</strong> $h_{\text{norm}} = \text{LN}_2(x')$ with parameters $\gamma_2, \beta_2$</li>
  <li><strong>Feed-Forward:</strong> $F = \text{GELU}(h_{\text{norm}} \cdot W_{fc} + b_{fc}) \cdot W_{\text{proj}} + b_{\text{proj}}$</li>
  <li><strong>Residual 2:</strong> $\text{output} = x' + F$</li>
</ol>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> tensor</li>
  <li>LayerNorm uses $\epsilon = 10^{-5}$</li>
  <li>Use the <a href="https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html" target="_blank">GELU tanh approximation</a>: $\text{GELU}(x) = 0.5\,x\!\left(1 + \tanh\!\left(\sqrt{\tfrac{2}{\pi}}\left(x + 0.044715\,x^3\right)\right)\right)$</li>
</ul>

<h2>Weight Layout</h2>
<p>All block parameters are packed into a single contiguous <code>weights</code> buffer
(7,087,872 floats) in the following order. Index into the buffer using the offsets below
(e.g. $W_{qkv}[i][j]$ is at <code>weights[1536 + i * 2304 + j]</code>).
All 2D matrices are stored in row-major order.</p>

<table style="border-collapse:separate; border-spacing:16px 8px;">
  <tr>
    <th style="text-align:left;">Parameter</th>
    <th style="text-align:left;">Shape</th>
    <th style="text-align:right;">Size</th>
    <th style="text-align:right;">Offset</th>
  </tr>
  <tr>
    <td>$\gamma_1$ (LN1 weight)</td>
    <td>(768,)</td>
    <td style="text-align:right;">768</td>
    <td style="text-align:right;">0</td>
  </tr>
  <tr>
    <td>$\beta_1$ (LN1 bias)</td>
    <td>(768,)</td>
    <td style="text-align:right;">768</td>
    <td style="text-align:right;">768</td>
  </tr>
  <tr>
    <td>$W_{qkv}$</td>
    <td>(768, 2304)</td>
    <td style="text-align:right;">1,769,472</td>
    <td style="text-align:right;">1,536</td>
  </tr>
  <tr>
    <td>$b_{qkv}$</td>
    <td>(2304,)</td>
    <td style="text-align:right;">2,304</td>
    <td style="text-align:right;">1,771,008</td>
  </tr>
  <tr>
    <td>$W_{\text{attn}}$</td>
    <td>(768, 768)</td>
    <td style="text-align:right;">589,824</td>
    <td style="text-align:right;">1,773,312</td>
  </tr>
  <tr>
    <td>$b_{\text{attn}}$</td>
    <td>(768,)</td>
    <td style="text-align:right;">768</td>
    <td style="text-align:right;">2,363,136</td>
  </tr>
  <tr>
    <td>$\gamma_2$ (LN2 weight)</td>
    <td>(768,)</td>
    <td style="text-align:right;">768</td>
    <td style="text-align:right;">2,363,904</td>
  </tr>
  <tr>
    <td>$\beta_2$ (LN2 bias)</td>
    <td>(768,)</td>
    <td style="text-align:right;">768</td>
    <td style="text-align:right;">2,364,672</td>
  </tr>
  <tr>
    <td>$W_{fc}$</td>
    <td>(768, 3072)</td>
    <td style="text-align:right;">2,359,296</td>
    <td style="text-align:right;">2,365,440</td>
  </tr>
  <tr>
    <td>$b_{fc}$</td>
    <td>(3072,)</td>
    <td style="text-align:right;">3,072</td>
    <td style="text-align:right;">4,724,736</td>
  </tr>
  <tr>
    <td>$W_{\text{proj}}$</td>
    <td>(3072, 768)</td>
    <td style="text-align:right;">2,359,296</td>
    <td style="text-align:right;">4,727,808</td>
  </tr>
  <tr>
    <td>$b_{\text{proj}}$</td>
    <td>(768,)</td>
    <td style="text-align:right;">768</td>
    <td style="text-align:right;">7,087,104</td>
  </tr>
</table>

<h2>Example</h2>
<p>With <code>seq_len</code> = 4, <code>x</code> uniformly drawn from [−1, 1], and weights randomly initialized
(see Weight Layout for the packing structure):</p>
<pre>
Input:  x.shape       = (4, 768)       # 4 token embeddings
        weights.shape = (7,087,872,)   # packed weight buffer
        seq_len       = 4
Output: output.shape  = (4, 768)       # transformed token embeddings
</pre>

<h2>Constraints</h2>
<ul>
  <li><code>d_model</code> = 768, <code>n_heads</code> = 12, <code>ffn_dim</code> = 3,072 (GPT-2 124M architecture)</li>
  <li>1 &le; <code>seq_len</code> &le; 4,096</li>
  <li>All tensors use 32-bit floating point</li>
  <li>Performance is measured with <code>seq_len</code> = 1,024</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// x, output, weights are device pointers
extern "C" void solve(const float* x, float* output, const float* weights, int seq_len) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# x, output, weights are tensors on the GPU
@cute.jit
def solve(
    x: cute.Tensor,
    output: cute.Tensor,
    weights: cute.Tensor,
    seq_len: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# x, weights are tensors on GPU
@jax.jit
def solve(x: jax.Array, weights: jax.Array, seq_len: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# x, output, weights are device pointers
@export
def solve(
    x: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    weights: UnsafePointer[Float32, MutExternalOrigin],
    seq_len: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# x, output, weights are tensors on the GPU
def solve(x: torch.Tensor, output: torch.Tensor, weights: torch.Tensor, seq_len: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# x, output, weights are tensors on the GPU
def solve(x: torch.Tensor, output: torch.Tensor, weights: torch.Tensor, seq_len: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
import math
from typing import Any, Dict, List

import torch
import torch.nn.functional as F

# GPT-2 124M fixed dimensions
D = 768
H = 12
DH = D // H  # 64
FFN = 3072

# Weight layout offsets in the packed buffer
O_LN1_W = 0
O_LN1_B = O_LN1_W + D
O_WQKV = O_LN1_B + D
O_BQKV = O_WQKV + D * 3 * D
O_WAPROJ = O_BQKV + 3 * D
O_BAPROJ = O_WAPROJ + D * D
O_LN2_W = O_BAPROJ + D
O_LN2_B = O_LN2_W + D
O_WFC = O_LN2_B + D
O_BFC = O_WFC + D * FFN
O_WPROJ = O_BFC + FFN
O_BPROJ = O_WPROJ + FFN * D
TOTAL_WEIGHTS = O_BPROJ + D


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="GPT-2 Transformer Block",
            atol=1e-03,
            rtol=1e-03,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        x: torch.Tensor,
        output: torch.Tensor,
        weights: torch.Tensor,
        seq_len: int,
    ):
        assert x.shape == (seq_len, D)
        assert output.shape == (seq_len, D)
        assert weights.shape == (TOTAL_WEIGHTS,)
        assert x.dtype == output.dtype == weights.dtype
        assert x.device.type == "cuda"
        assert output.device.type == "cuda"
        assert weights.device.type == "cuda"

        # unpack weights
        ln1_w = weights[O_LN1_W:O_LN1_B]
        ln1_b = weights[O_LN1_B:O_WQKV]
        W_qkv = weights[O_WQKV:O_BQKV].view(D, 3 * D)
        b_qkv = weights[O_BQKV:O_WAPROJ]
        W_attn = weights[O_WAPROJ:O_BAPROJ].view(D, D)
        b_attn = weights[O_BAPROJ:O_LN2_W]
        ln2_w = weights[O_LN2_W:O_LN2_B]
        ln2_b = weights[O_LN2_B:O_WFC]
        W_fc = weights[O_WFC:O_BFC].view(D, FFN)
        b_fc = weights[O_BFC:O_WPROJ]
        W_proj = weights[O_WPROJ:O_BPROJ].view(FFN, D)
        b_proj = weights[O_BPROJ : O_BPROJ + D]

        # layer norm 1
        x_norm = F.layer_norm(x, [D], ln1_w, ln1_b, eps=1e-5)

        # qkv projection
        qkv = x_norm @ W_qkv + b_qkv
        q, k, v = qkv.split(D, dim=-1)

        # reshape for multi-head attention: (H, seq_len, DH)
        q = q.view(seq_len, H, DH).transpose(0, 1)
        k = k.view(seq_len, H, DH).transpose(0, 1)
        v = v.view(seq_len, H, DH).transpose(0, 1)

        # scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(DH)
        attn_weights = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(attn_weights, v)

        # concat heads and project
        attn_out = attn_out.transpose(0, 1).contiguous().view(seq_len, D)
        attn_proj = attn_out @ W_attn + b_attn

        # residual connection 1
        hidden = x + attn_proj

        # layer norm 2
        h_norm = F.layer_norm(hidden, [D], ln2_w, ln2_b, eps=1e-5)

        # ffn: linear -> gelu (tanh approx) -> linear
        fc = h_norm @ W_fc + b_fc
        fc = F.gelu(fc, approximate="tanh")
        proj = fc @ W_proj + b_proj

        # residual connection 2
        output.copy_(hidden + proj)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "x": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "weights": (ctypes.POINTER(ctypes.c_float), "in"),
            "seq_len": (ctypes.c_int, "in"),
        }

    def _make_weights(self, device, dtype):
        scale = 0.02
        ln1_w = torch.empty(D, device=device, dtype=dtype).uniform_(0.8, 1.2)
        ln1_b = torch.empty(D, device=device, dtype=dtype).uniform_(-0.1, 0.1)
        W_qkv = torch.empty(D, 3 * D, device=device, dtype=dtype).normal_(0, scale)
        b_qkv = torch.zeros(3 * D, device=device, dtype=dtype)
        W_attn = torch.empty(D, D, device=device, dtype=dtype).normal_(0, scale)
        b_attn = torch.zeros(D, device=device, dtype=dtype)
        ln2_w = torch.empty(D, device=device, dtype=dtype).uniform_(0.8, 1.2)
        ln2_b = torch.empty(D, device=device, dtype=dtype).uniform_(-0.1, 0.1)
        W_fc = torch.empty(D, FFN, device=device, dtype=dtype).normal_(0, scale)
        b_fc = torch.zeros(FFN, device=device, dtype=dtype)
        W_proj = torch.empty(FFN, D, device=device, dtype=dtype).normal_(0, scale)
        b_proj = torch.zeros(D, device=device, dtype=dtype)
        return torch.cat(
            [
                ln1_w,
                ln1_b,
                W_qkv.flatten(),
                b_qkv,
                W_attn.flatten(),
                b_attn,
                ln2_w,
                ln2_b,
                W_fc.flatten(),
                b_fc,
                W_proj.flatten(),
                b_proj,
            ]
        )

    def _make_test_case(self, seq_len, zero_x=False):
        dtype = torch.float32
        device = "cuda"
        weights = self._make_weights(device, dtype)
        if zero_x:
            x = torch.zeros(seq_len, D, device=device, dtype=dtype)
        else:
            x = torch.empty(seq_len, D, device=device, dtype=dtype).uniform_(-1.0, 1.0)
        return {
            "x": x,
            "output": torch.empty(seq_len, D, device=device, dtype=dtype),
            "weights": weights,
            "seq_len": seq_len,
        }

    def generate_example_test(self) -> Dict[str, Any]:
        torch.manual_seed(0)
        return self._make_test_case(4)

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        tests = []
        # single token
        tests.append(self._make_test_case(1))
        # zero input
        tests.append(self._make_test_case(4, zero_x=True))
        # small edge cases
        tests.append(self._make_test_case(2))
        tests.append(self._make_test_case(4))
        # power-of-2
        tests.append(self._make_test_case(16))
        tests.append(self._make_test_case(64))
        # non-power-of-2
        tests.append(self._make_test_case(30))
        tests.append(self._make_test_case(100))
        # realistic
        tests.append(self._make_test_case(128))
        tests.append(self._make_test_case(256))
        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        return self._make_test_case(1024)


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
